In [0]:
print("Databricks Working")

Databricks Working


Check Catalog Access

In [0]:
spark.sql("SHOW CATALOGS").show()

+------------+
|     catalog|
+------------+
|datamodeling|
|insurance_uc|
|     samples|
|      system|
|   workspace|
+------------+



In [0]:
spark.sql("SHOW SCHEMAS").show()

+------------------+
|      databaseName|
+------------------+
|           default|
|              gold|
|information_schema|
+------------------+



AWS Connection Check

In [0]:
df = spark.read.parquet(
    "s3://pooja-airline-data-platform/athena/obt/"
)

print(df.count())

100000


Gold Layer

In [0]:
from pyspark.sql.functions import *

df = spark.read.parquet(
    "s3://pooja-airline-data-platform/athena/obt/"
)

print(f"Rows: {df.count()}")

Rows: 100000


Create Gold Catalog

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.gold
""")

DataFrame[]

In [0]:
spark.sql("SHOW SCHEMAS IN workspace").show()

+------------------+
|      databaseName|
+------------------+
|           default|
|              gold|
|information_schema|
+------------------+



GOLD TABLE 1

Revenue By Country

In [0]:
revenue_by_country = (
    df.groupBy("country")
      .agg(
          round(sum("ticket_price"), 2).alias("total_revenue"),
          count("*").alias("total_bookings"),
          round(avg("ticket_price"), 2).alias("avg_ticket_price")
      )
)

display(
    revenue_by_country.orderBy(
        col("total_revenue").desc()
    )
)

country,total_revenue,total_bookings,avg_ticket_price
Germany,1.31644541E7,12520,1051.47
USA,1.291738824E7,12215,1057.5
Japan,1.279307649E7,12178,1050.51
Australia,1.085823626E7,10442,1039.86
India,1.049669665E7,10011,1048.52
France,1.022796232E7,9862,1037.11
Singapore,9903747.62,9491,1043.49
UAE,9572595.96,9094,1052.63
UK,8741984.57,8307,1052.36
Canada,6212477.08,5880,1056.54


In [0]:
revenue_by_country.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.revenue_by_country")

In [0]:
spark.sql("""
SELECT *
FROM workspace.gold.revenue_by_country
ORDER BY total_revenue DESC
""").show()

+---------+-------------+--------------+----------------+
|  country|total_revenue|total_bookings|avg_ticket_price|
+---------+-------------+--------------+----------------+
|  Germany| 1.31644541E7|         12520|         1051.47|
|      USA|1.291738824E7|         12215|          1057.5|
|    Japan|1.279307649E7|         12178|         1050.51|
|Australia|1.085823626E7|         10442|         1039.86|
|    India|1.049669665E7|         10011|         1048.52|
|   France|1.022796232E7|          9862|         1037.11|
|Singapore|   9903747.62|          9491|         1043.49|
|      UAE|   9572595.96|          9094|         1052.63|
|       UK|   8741984.57|          8307|         1052.36|
|   Canada|   6212477.08|          5880|         1056.54|
+---------+-------------+--------------+----------------+



GOLD TABLE 2

Revenue By Airport

In [0]:
revenue_by_airport = (
    df.groupBy("airport_name")
      .agg(
          round(sum("ticket_price"), 2).alias("total_revenue"),
          count("*").alias("total_bookings")
      )
)

In [0]:
revenue_by_airport.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.revenue_by_airport")

In [0]:
spark.sql("""
SELECT *
FROM workspace.gold.revenue_by_airport
ORDER BY total_revenue DESC
""").show()

+--------------------+-------------+--------------+
|        airport_name|total_revenue|total_bookings|
+--------------------+-------------+--------------+
|Hyderabad Interna...|1.340984537E7|         12821|
|Ahmedabad Interna...|1.233603252E7|         11809|
|Chennai Internati...|1.101442313E7|         10523|
|Kolkata Internati...|1.096977939E7|         10464|
|Goa International...|1.068482687E7|         10231|
|Pune Internationa...|1.047588242E7|          9954|
|Delhi Internation...|   9616878.37|          9220|
|Bangalore Interna...|   9050389.16|          8564|
|Jaipur Internatio...|   8811369.83|          8342|
|Mumbai Internatio...|   8519192.23|          8072|
+--------------------+-------------+--------------+



GOLD TABLE 3

Flight Class Summary

In [0]:
flight_class_summary = (
    df.groupBy("flight_class")
      .agg(
          count("*").alias("total_bookings"),
          round(sum("ticket_price"), 2).alias("total_revenue"),
          round(avg("ticket_price"), 2).alias("avg_ticket_price")
      )
)

In [0]:
flight_class_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.flight_class_summary")

In [0]:
spark.sql("""
SELECT *
FROM workspace.gold.flight_class_summary
ORDER BY total_revenue DESC
""").show()

+---------------+--------------+-------------+----------------+
|   flight_class|total_bookings|total_revenue|avg_ticket_price|
+---------------+--------------+-------------+----------------+
|       Business|         25223|2.648448287E7|         1050.01|
|          First|         25042|2.637867887E7|         1053.38|
|        Economy|         25017|2.618337153E7|         1046.62|
|Premium Economy|         24718|2.584208602E7|         1045.48|
+---------------+--------------+-------------+----------------+



GOLD TABLE 4

Monthly Booking Trends

In [0]:
monthly_booking_trends = (
    df.groupBy(
        month("booking_date").alias("booking_month")
    )
    .agg(
        count("*").alias("total_bookings"),
        round(sum("ticket_price"), 2).alias("revenue")
    )
)

In [0]:
monthly_booking_trends.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.monthly_booking_trends")

In [0]:
spark.sql("""
SELECT *
FROM workspace.gold.monthly_booking_trends
ORDER BY booking_month
""").show()

+-------------+--------------+----------+
|booking_month|total_bookings|   revenue|
+-------------+--------------+----------+
|            1|          8778|9195175.96|
|            2|          7708| 8080388.5|
|            3|          8523|8927844.43|
|            4|          8287|8712306.36|
|            5|          8454|8822317.08|
|            6|          8160| 8584267.6|
|            7|          8435|8818738.31|
|            8|          8314|8755875.22|
|            9|          8236|8640426.84|
|           10|          8419|8820521.97|
|           11|          8293|8723094.37|
|           12|          8393|8807662.65|
+-------------+--------------+----------+



Final Validation

In [0]:
spark.sql("""
SHOW TABLES IN workspace.gold
""").show(truncate=False)

+--------+----------------------+-----------+
|database|tableName             |isTemporary|
+--------+----------------------+-----------+
|gold    |flight_class_summary  |false      |
|gold    |monthly_booking_trends|false      |
|gold    |revenue_by_airport    |false      |
|gold    |revenue_by_country    |false      |
+--------+----------------------+-----------+



In [0]:
display(revenue_by_country)

country,total_revenue,total_bookings,avg_ticket_price
Australia,1.085823626E7,10442,1039.86
Japan,1.279307649E7,12178,1050.51
USA,1.291738824E7,12215,1057.5
Singapore,9903747.62,9491,1043.49
Canada,6212477.08,5880,1056.54
India,1.049669665E7,10011,1048.52
UK,8741984.57,8307,1052.36
Germany,1.31644541E7,12520,1051.47
France,1.022796232E7,9862,1037.11
UAE,9572595.96,9094,1052.63


In [0]:
display(revenue_by_airport)

airport_name,total_revenue,total_bookings
Jaipur International Airport,8811369.83,8342
Ahmedabad International Airport,1.233603252E7,11809
Chennai International Airport,1.101442313E7,10523
Kolkata International Airport,1.096977939E7,10464
Bangalore International Airport,9050389.16,8564
Delhi International Airport,9616878.37,9220
Hyderabad International Airport,1.340984537E7,12821
Goa International Airport,1.068482687E7,10231
Mumbai International Airport,8519192.23,8072
Pune International Airport,1.047588242E7,9954


In [0]:
display(flight_class_summary)

flight_class,total_bookings,total_revenue,avg_ticket_price
First,25042,2.637867887E7,1053.38
Business,25223,2.648448287E7,1050.01
Premium Economy,24718,2.584208602E7,1045.48
Economy,25017,2.618337153E7,1046.62


In [0]:
display(monthly_booking_trends)

booking_month,total_bookings,revenue
12,8393,8807662.65
10,8419,8820521.97
5,8454,8822317.08
3,8523,8927844.43
1,8778,9195175.96
2,7708,8080388.5
9,8236,8640426.84
6,8160,8584267.6
7,8435,8818738.31
11,8293,8723094.37
